# Problemas de prueba de hipótesis

## Ejercicio 1

Usted es un nutricionista que investiga dos tipos diferentes de dietas para ver si existe una diferencia significativa en la pérdida de peso después de un mes.

| Diet 1 | Diet 2 |
|:-------|:-------|
| 2.0 | 3.0 |
| 2.5 | 3.2 |
| 3.0 | 3.1 |
| 2.8 | 2.9 |
| 2.3 | 2.8 |
| 2.7 | 3.0 |
| 2.5 | 3.2 |

### 1. Hipótesis

- **Hipótesis nula ($H_0$):** No existe diferencia en la pérdida de peso promedio entre las dos dietas → $\mu_1 = \mu_2$
- **Hipótesis alternativa ($H_1$):** Sí existe una diferencia significativa entre las dos dietas → $\mu_1 \neq \mu_2$

### 2. Prueba t de Student

Usamos `ttest_ind` porque tenemos **dos grupos independientes** (personas distintas en cada dieta).
El nivel de significancia es **α = 0.05**: si el p-value es menor, rechazamos $H_0$.

In [1]:
import scipy.stats as stats
import numpy as np

dieta1 = [2.0, 2.5, 3.0, 2.8, 2.3, 2.7, 2.5]
dieta2 = [3.0, 3.2, 3.1, 2.9, 2.8, 3.0, 3.2]

t_value, p_value = stats.ttest_ind(dieta1, dieta2)

print(f"t-value: {t_value}")
print(f"p-value: {p_value}")

t-value: -3.5383407969933938
p-value: 0.004083270191713912


### 3. Conclusión

El p-value obtenido (**0.0041**) es menor que α = 0.05, por lo tanto **rechazamos $H_0$**.

Esto significa que existe una diferencia estadísticamente significativa en la pérdida de peso entre las dos dietas.
La **Dieta 2** presenta una mayor pérdida de peso promedio que la Dieta 1.

---
## ANOVA

**ANOVA** (*Analysis of Variance*) es una técnica estadística para comparar las medias de **dos o más grupos**. Descompone la variabilidad total en dos componentes:

- **Variabilidad entre grupos:** diferencias entre las medias de cada grupo.
- **Variabilidad dentro del grupo:** dispersión de los datos dentro de cada grupo.

Las hipótesis en ANOVA son:
- **$H_0$:** Las medias de todos los grupos son iguales.
- **$H_1$:** Al menos una media de grupo es diferente.

Si el p-value es menor a 0.05, hay evidencia de que al menos un grupo difiere.

## Ejercicio 2

Un agricultor prueba tres fertilizantes distintos en 15 parcelas (5 por fertilizante) para ver cuál produce más maíz.

| Fertilizante 1 | Fertilizante 2 | Fertilizante 3 |
|:-------------|:-------------|:-------------|
| 20 | 22 | 24 |
| 21 | 21 | 23 |
| 20 | 23 | 22 |
| 19 | 22 | 23 |
| 20 | 21 | 24 |

### 1. Hipótesis

- **Hipótesis nula ($H_0$):** Los tres fertilizantes producen el mismo rendimiento promedio → $\mu_1 = \mu_2 = \mu_3$
- **Hipótesis alternativa ($H_1$):** Al menos un fertilizante produce un rendimiento diferente

### 2. Prueba ANOVA

Usamos `f_oneway` de scipy que recibe los grupos y devuelve el **estadístico F** y el **p-value**.
Un F grande indica que la variabilidad entre grupos supera la variabilidad dentro de los grupos.

In [2]:
!pip install statsmodels -q

import scipy.stats as stats
import numpy as np

fertilizante1 = [20, 21, 20, 19, 20]
fertilizante2 = [22, 21, 23, 22, 21]
fertilizante3 = [24, 23, 22, 23, 24]

f_value, p_value = stats.f_oneway(fertilizante1, fertilizante2, fertilizante3)

print(f"f-value: {f_value}")
print(f"p-value: {p_value}")

f-value: 20.31578947368421
p-value: 0.000140478247931904


### 3. Conclusión ANOVA

El p-value (**0.00014**) es menor que α = 0.05, por lo tanto **rechazamos $H_0$**.

Existe una diferencia significativa en el rendimiento promedio entre al menos dos fertilizantes.
Sin embargo, ANOVA **no nos dice cuál** fertilizante es diferente — para eso necesitamos una prueba post-hoc.

### 4. ¿Cuál fertilizante es mejor? → Prueba post-hoc de Tukey HSD

Después de realizar el ANOVA, se puede ver que existen diferencias significativas entre los grupos. El siguiente paso es identificar qué fertilizante es mejor, para lo cual realizaremos una prueba post-hoc. Una de estas pruebas es la prueba de **Tukey HSD**, que se basa en lo siguiente:

1. Si el primer fertilizante es significativamente diferente del segundo, pero **no** hay diferencia significativa entre el primero y el tercero, ni entre el segundo y el tercero, podríamos concluir que el primer fertilizante es el mejor o el peor, dependiendo de la dirección de la diferencia.

2. Si **todos** los fertilizantes son significativamente diferentes entre sí, comparamos las medias para determinar cuál es el mejor.

---

Cómo interpretar la tabla de resultados:
- **`reject = True`** → ese par de grupos **sí** tiene diferencia significativa.
- **`reject = False`** → ese par de grupos **no** tiene diferencia significativa.
- **`meandiff` positivo** → `group2` tiene mayor media que `group1`.
- **`meandiff` negativo** → `group1` tiene mayor media que `group2`.

In [3]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

data = np.concatenate([fertilizante1, fertilizante2, fertilizante3])
labels = ["F1"] * 5 + ["F2"] * 5 + ["F3"] * 5

resultado_tukey = pairwise_tukeyhsd(data, labels, alpha=0.05)
print(resultado_tukey)

Multiple Comparison of Means - Tukey HSD, FWER=0.05
group1 group2 meandiff p-adj  lower  upper  reject
--------------------------------------------------
    F1     F2      1.8 0.0099 0.4572 3.1428   True
    F1     F3      3.2 0.0001 1.8572 4.5428   True
    F2     F3      1.4 0.0409 0.0572 2.7428   True
--------------------------------------------------


### 5. Conclusión final

Esta tabla es muy sencilla de interpretar. Miramos la columna `reject`: si el valor es `True`, significa que hay una diferencia estadísticamente significativa entre esos dos grupos. Si hay diferencia significativa, miramos la columna `meandiff` para ver cuánto difieren las medias.

En este caso, **todos los pares tienen `reject = True`**, lo que significa que los tres fertilizantes son significativamente diferentes entre sí. Comparando las medias:

| Fertilizante | Media (kg) |
|:---|:---:|
| Fertilizante 1 | 20.0 |
| Fertilizante 2 | 21.8 |
| Fertilizante 3 | 23.2 |

Por tanto, **el Fertilizante 3 es el más efectivo** para la producción de maíz, seguido del Fertilizante 2 y por último el Fertilizante 1.